# Biểu đồ đánh giá DRL (PPO/SAC) cho báo cáo đồ án

Phiên bản notebook của `plot_metrics.py` — dùng khi muốn xem trước từng biểu đồ ngay trong Jupyter
trước khi nhúng vào báo cáo, thay vì chạy CLI rồi mở file ảnh riêng. Hai bản **dùng chung toàn bộ logic**
(hàm vẽ, bảng màu, style) từ `plot_metrics.py` — notebook này chỉ gọi lại các hàm đó với `show=True`
để hiển thị inline, không viết lại bất kỳ phép vẽ nào — sửa logic vẫn chỉ sửa ở một nơi.

**Không cần CARLA/torch** — chỉ `numpy` + `matplotlib`, chạy được trên máy viết báo cáo (khác máy train).

**Cách dùng**: mở notebook này từ thư mục `drl_training/` (để import `plot_metrics.py` cạnh nó), sửa
đường dẫn ở cell **Cấu hình** bên dưới cho khớp thư mục `runs/...` thật của bạn, rồi **Run All**. Mỗi
biểu đồ vừa hiện inline vừa được lưu ra `--output` (`.png` cho Word + `.pdf` vector cho LaTeX/Overleaf).

Chỉ có 1 thuật toán (ví dụ mới train PPO)? Để trống `SAC_DIR`/`EVAL_SAC_CSV` = `None` — các cell
liên quan SAC/so sánh 2 thuật toán sẽ tự in cảnh báo và bỏ qua, không lỗi.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import numpy as np

# Gia dinh notebook duoc mo/chay tu thu muc drl_training/ (canh plot_metrics.py). Neu
# khong, sua NOTEBOOK_DIR thanh duong dan tuyet doi toi drl_training/.
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from plot_metrics import (  # noqa: E402 - tai su dung toan bo logic ve tu plot_metrics.py
    COLOR_PPO, COLOR_SAC, COLOR_IL, Run, IlDemo, _apply_style,
    plot_il_closed_loop, plot_il_lane_keeping,
    plot_learning_curve, plot_update_diagnostic, plot_terminate_reason,
    plot_eval_comparison, plot_throughput,
)

_apply_style()
print("Da nap plot_metrics.py tu:", NOTEBOOK_DIR / "plot_metrics.py")

## Cấu hình — sửa các đường dẫn này cho khớp lần train thật của bạn

`PPO_DIR`/`SAC_DIR` = thư mục `output` của `train_ppo.py`/`train_sac.py` (chứa `episode_log.csv` +
`update_log.csv`). `EVAL_*_CSV` = file từ `evaluate.py --eval-csv-out ...` (tùy chọn, chỉ cần cho
biểu đồ so sánh cuối). `IL_MAE`/`IL_COLLISION_RATE` = số liệu riêng của bước IL (mục 11 notebook IL),
dùng làm đường baseline tham chiếu — để `None` nếu chưa có.

In [ ]:
PPO_DIR = "runs/ppo_lane_keep"          # None neu chua train PPO
SAC_DIR = "runs/sac_lane_keep"          # None neu chua train SAC

EVAL_PPO_CSV = "runs/ppo_lane_keep/eval_results.csv"   # None neu chua chay evaluate.py --eval-csv-out
EVAL_SAC_CSV = "runs/sac_lane_keep/eval_results.csv"   # None neu chua chay evaluate.py --eval-csv-out

# Baseline IL vong kin — ket qua demo_il.py. Dat NHIEU dong de so sanh ban do.
# Cac gia tri baseline trong bieu do 09 duoc suy ra TU DAY, khong con go tay nua.
IL_DEMO_CSVS = {
    "Town03": "runs/il_demo_v9_fixed/il_demo_results.csv",
    "Town04": "runs/il_demo_v9_town04/il_demo_results.csv",
}

# Ghi de thu cong (de None de dung so suy ra tu IL_DEMO_CSVS o tren).
IL_MAE = None              # lech lan trung binh |m| cua buoc IL, tinh tren DUONG THUONG
IL_COLLISION_RATE = None   # ti le va cham (%) cua buoc IL

REWARD_WINDOW = 20   # do rong cua so trung binh truot cho duong hoc; tu thu nho neu it episode
# Cung thu muc voi lenh CLI trong README (`--output ../report_figures`) — de hai
# duong chay khong sinh ra hai bo hinh o hai noi khac nhau.
OUTPUT_DIR = Path("../report_figures").expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
runs = [
    Run("PPO", COLOR_PPO, run_dir=PPO_DIR, eval_csv=EVAL_PPO_CSV),
    Run("SAC", COLOR_SAC, run_dir=SAC_DIR, eval_csv=EVAL_SAC_CSV),
]
runs = [r for r in runs if r.episode_rows or r.update_rows or r.eval_rows]
ppo_run = next((r for r in runs if r.name == "PPO" and r.update_rows), None)
sac_run = next((r for r in runs if r.name == "SAC" and r.update_rows), None)

il_demos = [IlDemo(label, path) for label, path in IL_DEMO_CSVS.items()]
il_demos = [d for d in il_demos if d.ok]

# Suy ra baseline IL cho bieu do 09. Dung so do CHI DUONG THUONG de cung mot thang do voi
# cot `mean_abs_lane_offset_road` ma evaluate.py ghi cho PPO/SAC — neu lay so lan nga tu thi
# hai ben do bang hai thuoc khac nhau va ket luan se sai.
il_mae, il_collision_rate, il_reward = IL_MAE, IL_COLLISION_RATE, None
if il_demos:
    if il_mae is None:
        il_mae = float(np.nanmean([np.nanmean(d.col("mean_abs_offset_road")) for d in il_demos]))
    if il_collision_rate is None:
        il_collision_rate = 100.0 * float(np.mean(np.concatenate([d.collided for d in il_demos])))
    _r = np.concatenate([d.col("reward") for d in il_demos])
    il_reward = (float(np.nanmean(_r)), float(np.nanstd(_r)))

if not runs and not il_demos:
    print("[!] Khong doc duoc du lieu nao — kiem tra lai duong dan o cell Cau hinh.")
else:
    print("Lan chay DRL doc duoc :", ", ".join(r.name for r in runs) or "(chua co)")
    print("Baseline IL doc duoc  :", ", ".join(d.label for d in il_demos) or "(chua co)")
print("Bieu do se duoc luu vao:", OUTPUT_DIR)

## 0. Baseline IL vòng kín — mốc "trước khi có DRL"

Đây là kết quả `demo_il.py`: policy IL chạy trên CARLA thật, **chưa học thêm gì**. Mọi con số DRL
phía dưới đều phải đối chiếu về đây, nếu không thì không nói được DRL cải thiện bao nhiêu.

Hình thứ hai là hình quan trọng nhất về mặt phương pháp: trong ngã tư, `lane_offset_m` được suy ra
từ "làn đường gần nhất", mà các nhánh cắt nhau nên tham chiếu đó nhảy sang nhánh vuông góc chỉ sau
vài mét — con số vô nghĩa. Gộp nó vào trung bình làm năng lực bám làn **trông tệ hơn thực tế**.

In [ ]:
if il_demos:
    plot_il_closed_loop(il_demos, "00_il_closed_loop", OUTPUT_DIR, show=True)
    plot_il_lane_keeping(il_demos, "00b_il_lane_keeping", OUTPUT_DIR, show=True)
else:
    print("Chua co il_demo_results.csv — chay demo_il.py truoc (xem IL_DEMO_CSVS o cell Cau hinh).")

## 1. Đường học — Reward theo episode

Nét nhạt = reward thô từng episode, nét đậm = trung bình trượt `REWARD_WINDOW` episode gần nhất.

In [ ]:
plot_learning_curve(runs, "episode_reward", "Reward / episode", "Duong hoc — Reward theo episode",
                     "01_reward_curve", OUTPUT_DIR, REWARD_WINDOW, show=True)

## 2. Đường học — Độ dài episode

Episode dài dần theo thời gian train thường là dấu hiệu policy ít va chạm/lệch làn sớm hơn.

In [ ]:
plot_learning_curve(runs, "episode_len", "So buoc / episode", "Duong hoc — Do dai episode",
                     "02_episode_length", OUTPUT_DIR, REWARD_WINDOW, show=True)

## 3. Lý do kết thúc episode theo tiến trình train

Cột xếp chồng 100% theo từng “cửa sổ” episode liên tiếp — muốn thấy phần **đỏ (va chạm)** thu hẹp và
phần **xanh (hết giờ, an toàn)** mở rộng dần về cuối quá trình train.

In [ ]:
plot_terminate_reason(runs, "03_terminate_reason", OUTPUT_DIR, show=True)

## 4–5. Chẩn đoán PPO (clipped surrogate)

`approx_kl` nên dời quanh `target_kl=0.02` (đường gạch ngang) — liên tục vượt xa nguỡng này là dấu
hiệu learning rate quá cao. Bỏ qua nếu chưa train PPO.

In [ ]:
if ppo_run:
    plot_update_diagnostic([ppo_run],
        [("policy_loss", "Policy loss", None), ("value_loss", "Value loss", None)],
        "PPO — Policy loss & Value loss", "04_ppo_losses", OUTPUT_DIR, REWARD_WINDOW, show=True)

    TARGET_KL = 0.02
    plot_update_diagnostic([ppo_run],
        [("approx_kl", "Approx. KL", (TARGET_KL, "target_kl=%.2f" % TARGET_KL)),
         ("clip_fraction", "Ti le bi clip", None),
         ("entropy", "Entropy chinh sach", None)],
        "PPO — Chan doan huan luyen", "05_ppo_diagnostics", OUTPUT_DIR, REWARD_WINDOW, show=True)
else:
    print("[!] Chua co du lieu PPO (PPO_DIR trong hoac thieu update_log.csv) — bo qua.")

## 6–7. Chẩn đoán SAC (twin-Q + auto temperature)

`alpha` (temperature) thường giảm dần khi chính sách ổn định; `mean_q` tăng bất thường (rất lớn hoặc
âm sâu) là dấu hiệu overestimation. Bỏ qua nếu chưa train SAC.

In [ ]:
if sac_run:
    plot_update_diagnostic([sac_run],
        [("critic_loss", "Critic loss", None), ("actor_loss", "Actor loss", None)],
        "SAC — Critic loss & Actor loss", "06_sac_losses", OUTPUT_DIR, REWARD_WINDOW, show=True)

    plot_update_diagnostic([sac_run],
        [("alpha", "Temperature (alpha)", None), ("mean_q", "Mean Q", None),
         ("entropy", "Entropy chinh sach", None)],
        "SAC — Chan doan huan luyen", "07_sac_diagnostics", OUTPUT_DIR, REWARD_WINDOW, show=True)
else:
    print("[!] Chua co du lieu SAC (SAC_DIR trong hoac thieu update_log.csv) — bo qua.")

## 8. Thông lượng huấn luyện

`steps_per_sec` — biểu đồ phụ trợ đánh giá hiệu năng hệ thống (không phải chất lượng policy).

In [ ]:
plot_throughput(runs, "08_throughput", OUTPUT_DIR, REWARD_WINDOW, show=True)

## 9. So sánh đánh giá cuối (PPO vs SAC vs IL)

Cần `EVAL_PPO_CSV`/`EVAL_SAC_CSV` (từ `evaluate.py --eval-csv-out ... --deterministic`). `IL_MAE`/
`IL_COLLISION_RATE` chỉ xuất hiện nếu được điền ở cell Cấu hình.

In [ ]:
plot_eval_comparison(runs, il_mae, il_collision_rate, il_reward,
                     "09_eval_comparison", OUTPUT_DIR, show=True)

## 10. Bảng số liệu — chép thẳng vào báo cáo

Cell dưới in ra đúng những con số cần cho phần kết quả. Cột `|lệch làn|` của **cả ba** đều là số đo
trên đường thường (đã bỏ ngã tư), nên so sánh trực tiếp được.

In [ ]:
def _fmt(v, spec="%.3f"):
    return "--" if v is None or (isinstance(v, float) and not np.isfinite(v)) else spec % v

print("=" * 78)
print("BASELINE IL (demo_il.py, vong kin, chua co DRL)")
print("=" * 78)
if il_demos:
    print("  %-10s %10s %12s %10s %10s %9s" % ("ban do", "q.duong", "|lech| duong", "off_lane", "va cham", "nga tu"))
    for d in il_demos:
        n = len(d.rows)
        print("  %-10s %9.0fm %11s m %9.1f%% %8.0f%% %8.0f%%" % (
            d.label,
            np.nanmean(d.col("distance_m")),
            _fmt(float(np.nanmean(d.col("mean_abs_offset_road")))),
            100.0 * np.nansum(d.col("off_lane_steps")) / max(np.nansum(d.col("steps")), 1),
            100.0 * d.collided.mean(),
            100.0 * d.junction_rate))
    print("  --> dung lam moc: |lech| %s m | va cham %.0f%% | reward %.0f" % (
        _fmt(il_mae), il_collision_rate, il_reward[0]))
else:
    print("  (chua co du lieu)")

print()
print("=" * 78)
print("DANH GIA CUOI (evaluate.py --deterministic)")
print("=" * 78)
_eval = [r for r in runs if r.eval_rows]
if _eval:
    print("  %-6s %10s %10s %14s %10s" % ("t.toan", "reward", "va cham", "|lech| duong", "so ep"))
    for r in _eval:
        rew = np.array([float(x["reward"]) for x in r.eval_rows])
        col = np.array([x["collided"] in ("True", "true", "1") for x in r.eval_rows])
        off = np.array([float(x.get("mean_abs_lane_offset_road", "nan") or "nan")
                        for x in r.eval_rows])
        if not np.isfinite(off).any():        # file eval cu, truoc khi tach nga tu
            off = np.array([float(x["mean_abs_lane_offset"]) for x in r.eval_rows])
            print("  [!] %s: eval CSV cu, |lech| con lan nga tu — khong so truc tiep voi IL duoc." % r.name)
        print("  %-6s %9.0f %9.0f%% %13s m %9d" % (
            r.name, rew.mean(), 100.0 * col.mean(), _fmt(float(np.nanmean(off))), len(rew)))
        if il_mae is not None and np.isfinite(np.nanmean(off)):
            print("         so voi IL: reward %+.0f%% | va cham %+.0f diem%% | |lech| %+.0f%%" % (
                100.0 * (rew.mean() / il_reward[0] - 1.0),
                100.0 * col.mean() - il_collision_rate,
                100.0 * (float(np.nanmean(off)) / il_mae - 1.0)))
else:
    print("  (chua co eval CSV — chay evaluate.py voi --eval-csv-out)")
print("=" * 78)

## Tổng kết

Mỗi biểu đồ ở trên đã được lưu vào `OUTPUT_DIR` ở cả `.png` (nhúng vào Word) và `.pdf` (vector, nhúng vào
LaTeX/Overleaf). Danh sách file:

In [ ]:
for path in sorted(OUTPUT_DIR.glob("*.png")):
    print(path.name)